In [2]:
import math
from collections import deque
import heapq

# --- Helper classes and functions ---

class ReservoirProblem:
    """Encapsulates the reservoir problem's parameters and logic."""
    def __init__(self, capacities, initial_state, target_state):
        self.capacities = capacities
        self.initial_state = tuple(initial_state)
        self.target_state = tuple(target_state)
        # Safety thresholds are 20% of capacity
        self.safety_thresholds = [0.2 * c for c in capacities]

    def get_successors(self, state):
        """Generates all valid next states from the current state."""
        successors = []
        num_reservoirs = len(self.capacities)

        for i in range(num_reservoirs):  # Source reservoir index
            for j in range(num_reservoirs):  # Destination reservoir index
                if i == j:
                    continue

                # Unpack current state
                r = list(state)
                
                # Calculate maximum water the source can give
                max_give = r[i] - self.safety_thresholds[i]
                if max_give <= 0:
                    continue

                # Calculate maximum water the destination can accept
                max_accept = self.capacities[j] - r[j]
                if max_accept <= 0:
                    continue
                
                # Transfer the minimum of the two amounts
                transfer_amount = min(max_give, max_accept)

                # Create the new state
                new_state = list(state)
                new_state[i] -= transfer_amount
                new_state[j] += transfer_amount
                
                action = f"Open valve ({i+1}->{j+1})"
                successors.append((action, tuple(new_state)))
                
        return successors

    def is_goal(self, state):
        """Checks if a state is the target state."""
        return all(math.isclose(state[i], self.target_state[i]) for i in range(len(state)))

def heuristic_mismatched_reservoirs(state, target_state):
    """
    A* heuristic: ceil(number of mismatched reservoirs / 2).
    A single move can fix at most two reservoirs.
    """
    mismatched_count = sum(1 for i in range(len(state)) if not math.isclose(state[i], target_state[i]))
    return math.ceil(mismatched_count / 2)

# --- Main Search Algorithm ---

def search(problem, method='bfs'):
    """
    Performs BFS, DFS, or A* search on the given problem.

    Arguments:
        problem: An instance of the ReservoirProblem class.
        method: A string, one of 'bfs', 'dfs', 'astar'.

    Returns:
        A list of actions representing the solution path, or None if no solution is found.
    """
    initial_state = problem.initial_state
    
    if method == 'bfs':
        fringe = deque([(initial_state, [])])  # Queue: (state, path)
    elif method == 'dfs':
        fringe = [(initial_state, [])]  # Stack: (state, path)
    elif method == 'astar':
        h_cost = heuristic_mismatched_reservoirs(initial_state, problem.target_state)
        fringe = [(h_cost, 0, initial_state, [])]  # Priority Queue: (f_cost, g_cost, state, path)
        heapq.heapify(fringe)
    
    # Dictionary to handle visited states and re-visiting with lower cost
    visited = {initial_state: 0}

    while fringe:
        # --- Pop from fringe based on search method ---
        if method == 'bfs':
            current_state, path = fringe.popleft()
            g_cost = len(path)
        elif method == 'dfs':
            current_state, path = fringe.pop()
            g_cost = len(path)
        elif method == 'astar':
            f_cost, g_cost, current_state, path = heapq.heappop(fringe)
            # If we found a shorter path to an already expanded node, skip
            if g_cost > visited[current_state]:
                continue

        # --- Goal Check ---
        if problem.is_goal(current_state):
            return path

        # --- Generate Successors ---
        for action, next_state in problem.get_successors(current_state):
            new_g_cost = g_cost + 1
            
            # If we haven't seen this state or found a cheaper path to it
            if next_state not in visited or new_g_cost < visited[next_state]:
                visited[next_state] = new_g_cost
                new_path = path + [action]
                
                # --- Push to fringe based on search method ---
                if method == 'bfs':
                    fringe.append((next_state, new_path))
                elif method == 'dfs':
                    fringe.append((next_state, new_path))
                elif method == 'astar':
                    h_cost = heuristic_mismatched_reservoirs(next_state, problem.target_state)
                    f_cost = new_g_cost + h_cost
                    heapq.heappush(fringe, (f_cost, new_g_cost, next_state, new_path))
                    
    return None # No solution found

def print_solution(solution, method_name):
    """Prints the solution in the required format."""
    print(f"--- Solution found using {method_name} ---")
    if solution is None:
        print("No solution found.")
    else:
        for step in solution:
            print(step)
        print(f"Number of valve operations = {len(solution)}")
    print("-" * 35 + "\n")


# --- Main execution block ---
if __name__ == "__main__":
    # --- Input ---
    capacities_str = "8.0 5.0 3.0".split()
    initial_str = "8.0 0.0 0.0".split()
    target_str = "2.4 5.0 0.6".split()

    # Convert inputs to float lists/tuples
    capacities = [float(c) for c in capacities_str]
    initial_water = [float(i) for i in initial_str]
    target_water = [float(t) for t in target_str]

    # Create the problem instance
    problem = ReservoirProblem(capacities, initial_water, target_water)

    # Solve using BFS
    solution_bfs = search(problem, method='bfs')
    print_solution(solution_bfs, "BFS")
    
    # Solve using DFS
    solution_dfs = search(problem, method='dfs')
    print_solution(solution_dfs, "DFS")

    # Solve using A*
    solution_astar = search(problem, method='astar')
    print_solution(solution_astar, "A* Search")

--- Solution found using BFS ---
Open valve (1->2)
Open valve (1->3)
Open valve (3->1)
Number of valve operations = 3
-----------------------------------

--- Solution found using DFS ---
Open valve (1->3)
Open valve (3->2)
Open valve (2->3)
Open valve (1->2)
Open valve (3->2)
Open valve (3->1)
Number of valve operations = 6
-----------------------------------

--- Solution found using A* Search ---
Open valve (1->2)
Open valve (1->3)
Open valve (3->1)
Number of valve operations = 3
-----------------------------------

